# CMT — large300 — Nouveaux objectifs de stage

1. **Visualisation** à un temps donné (+ évolution) : flux de Reynolds, cisaillement, $u$, $w$, $\partial_z u$, $\partial_z^2 u$.
2. **Equation discovery (STLSQ)** avec des termes physiques *grande échelle* uniquement.
3. **Robustesse temporelle** du split train/test (walk-forward vs split aléatoire).
4. **$\psi$ moyennée en $y$ → POD** : combien de modes garder, reconstruction de $u,w$.
5. **Circulation** — cellules fermées de $\psi$ à signe unique.

Setup identique à d'habitude : lecture niveau par niveau, $\rho_0$ via $T_v$, masques PRW, pas de dépendance nouvelle (numpy / scipy / xarray / matplotlib seulement).


## 0. Imports & configuration

Identique au pipeline **large300** habituel : fichiers 3D séparés par variable dans `3D/MESONH_RCE_large300_3D_{var}.nc` (variables `ua`,`va`,`wa`,`ta`,`pa`,`hus`), altitude reprise du 1D, état stationnaire = **dernier tiers** des pas de temps, $\rho_0(z)$ via la température virtuelle. Lecture **niveau par niveau** pour la RAM.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.optimize import curve_fit
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve
from scipy.ndimage import label as ndi_label

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.25,
    'image.cmap': 'RdBu_r',
})

# ============================================================
#  CONFIGURATION  (identique au notebook rcemip_large300)
# ============================================================
DIR_3D = '3D';  DIR_2D = '2D';  DIR_1D = '1D'
def path3d(var): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')
def path2d(var): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{var}.nc')
def path1d(var): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{var}.nc')

Rd, Rv = 287.05, 461.5
EPSILON = Rd / Rv          # ~ 0.622
BLOC    = 2                # taille de bloc temporel (RAM)

print('Config prête.')


In [ ]:

# --- (0a) Métadonnées : dims, tailles, altitude, fenêtre stationnaire ---
_ds = xr.open_dataset(path3d('ua'));  _da = _ds['ua']
dim_t, dim_z, dim_y, dim_x = _da.dims        # ordre (t, z, y, x)
n_t = _da.sizes[dim_t];  n_z = _da.sizes[dim_z]
n_y = _da.sizes[dim_y];  n_x = _da.sizes[dim_x]
_ds.close();  del _ds, _da;  gc.collect()

_t1 = xr.open_dataset(path1d('ua_avg'))
alt = _t1['altitude'].values.astype(float).copy()
_t1.close()

t_stat   = int(2 * n_t / 3)          # dernier tiers = stationnaire
idx_stat = slice(t_stat, None)
n_stat   = n_t - t_stat
print(f'Grille : {n_t} t x {n_z} z x {n_y} y x {n_x} x')
print(f'Altitude : {alt[0]:.0f} -> {alt[-1]:.0f} m')
print(f'Stationnaire : t={t_stat}->{n_t-1} ({n_stat} pas)')

# grille horizontale (m) — dx/dy = 1000 m par défaut si coords = indices
try:
    xcoord = _ds.coords[dim_x].values.astype(float)
    dx = float(xcoord[1] - xcoord[0])
    if dx < 10:  # coords = indices entiers -> fallback
        raise ValueError
except Exception:
    dx = 1000.0
dy = dx
x = np.arange(n_x) * dx
print(f'dx = dy = {dx:.0f} m')


In [ ]:

# --- (0b) Profil rho_0(z) via température virtuelle (gaz parfaits) ---
rho0 = np.zeros(n_z)
ds_ta  = xr.open_dataset(path3d('ta'))
ds_pa  = xr.open_dataset(path3d('pa'))
ds_hus = xr.open_dataset(path3d('hus'))
for iz in range(n_z):
    ta_z  = ds_ta['ta'].isel({dim_z: iz, dim_t: idx_stat}).values
    pa_z  = ds_pa['pa'].isel({dim_z: iz, dim_t: idx_stat}).values
    hus_z = ds_hus['hus'].isel({dim_z: iz, dim_t: idx_stat}).values
    tv = ta_z * (1.0 + (1.0 / EPSILON - 1.0) * hus_z)
    rho0[iz] = np.mean(pa_z / (Rd * tv))
ds_ta.close(); ds_pa.close(); ds_hus.close(); gc.collect()

fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(rho0, alt / 1000)
ax.set_xlabel(r'$\rho_0$ (kg/m$^3$)'); ax.set_ylabel('z (km)')
ax.set_title(r'Profil $\rho_0(z)$')
fig.tight_layout(); plt.show()


In [ ]:

# --- (0c) Masques humide / sec (PRW), 2D (y,x), seuil = médiane ---
ds_prw = xr.open_dataset(path2d('prw'))
prw_t  = ds_prw['prw'].isel({dim_t: idx_stat}).values     # (n_stat, ny, nx)
ds_prw.close(); gc.collect()
prw = prw_t.mean(axis=0)                                   # moyenne temporelle (ny,nx)
seuil_prw = np.median(prw)
mh = prw > seuil_prw     # humide
ms = ~mh                 # sec
print(f'Seuil PRW = {seuil_prw:.1f} kg/m2  -  {mh.mean()*100:.0f}% de colonnes humides')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.pcolormesh(prw, cmap='YlGnBu', shading='auto')
ax.contour(mh.astype(float), levels=[0.5], colors='k', linewidths=1)
ax.set_title('PRW moyen + contour humide/sec')
fig.colorbar(im, ax=ax, label='PRW (kg/m2)')
fig.tight_layout(); plt.show()


## 1. Visualisation — champs à un temps donné + évolution temporelle

Flux de Reynolds $\rho_0\langle u'w'\rangle$, cisaillement $\partial_z\bar u$, $\bar u$, $\bar w$, $\partial_z\bar u$, $\partial_z^2\bar u$ — à un instant donné et en évolution (Hovmöller $z$-$t$).

In [ ]:

# --- Profils rho0<u'w'>(z,t), ubar(z,t), wbar(z,t), niveau par niveau ---
flux  = np.zeros((n_z, n_stat))
ubar  = np.zeros((n_z, n_stat))
wbar  = np.zeros((n_z, n_stat))

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    u = ds_u['ua'].isel({dim_z: iz, dim_t: idx_stat}).values   # (n_stat, ny, nx)
    w = ds_w['wa'].isel({dim_z: iz, dim_t: idx_stat}).values
    ub = u.mean(axis=(1, 2))
    wb = w.mean(axis=(1, 2))
    up = u - ub[:, None, None]
    wp = w - wb[:, None, None]
    flux[iz, :] = rho0[iz] * (up * wp).mean(axis=(1, 2))
    ubar[iz, :] = ub
    wbar[iz, :] = wb
ds_u.close(); ds_w.close(); gc.collect()

dudz   = np.gradient(ubar, alt, axis=0)
d2udz2 = np.gradient(dudz, alt, axis=0)
print('flux, ubar, wbar, dudz, d2udz2 prêts :', flux.shape)


In [ ]:

def dashboard_at_time(it):
    '''Panneau de profils verticaux a l instant it (indice dans la fenetre stationnaire).'''
    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)
    panels = [
        (flux[:, it],   r"$\rho_0\langle u'w'\rangle$"),
        (dudz[:, it],   r"$\partial_z \bar u$"),
        (ubar[:, it],   r"$\bar u$ (m/s)"),
        (wbar[:, it],   r"$\bar w$ (m/s)"),
        (d2udz2[:, it], r"$\partial_z^2 \bar u$"),
    ]
    for ax, (field, label) in zip(axes, panels):
        ax.plot(field, alt / 1000, lw=1.5)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel('z (km)')
    fig.suptitle(f'Diagnostics — instant {it} (t={t_stat+it})')
    fig.tight_layout()
    plt.show()

dashboard_at_time(n_stat // 2)


In [ ]:

def hovmoller(field, title, cmap='RdBu_r'):
    vmax = np.nanpercentile(np.abs(field), 98) + 1e-30
    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.pcolormesh(np.arange(n_stat), alt / 1000, field,
                        cmap=cmap, vmin=-vmax, vmax=vmax, shading='auto')
    ax.set_xlabel('indice temporel (fenêtre stationnaire)')
    ax.set_ylabel('z (km)')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()

hovmoller(flux, r"$\rho_0\langle u'w'\rangle$ (z,t)")
hovmoller(dudz, r"$\partial_z \bar u$ (z,t)")


## 2. Equation discovery — termes physiques grande échelle uniquement

Bibliothèque de candidats : $\{\bar u,\ \partial_z\bar u,\ \partial_z^2\bar u,\ \bar w,\ \bar u\partial_z\bar u,\ z\partial_z\bar u,\ \bar u^2\}$ — uniquement des quantités grande échelle, pas de termes de sous-maille. STLSQ codé à la main (déjà la méthode utilisée dans le notebook `6_CMT_trois_objectifs`), pas de nouvelle dépendance.

In [ ]:

def build_library():
    Z = np.tile(alt[:, None], (1, n_stat))
    terms = {
        'u':      ubar,
        'dudz':   dudz,
        'd2udz2': d2udz2,
        'w':      wbar,
        'u_dudz': ubar * dudz,
        'z_dudz': Z * dudz,
        'u2':     ubar ** 2,
    }
    names = list(terms.keys())
    Theta = np.stack([terms[n].ravel() for n in names], axis=1)
    return Theta, names


def nse(y_true, y_pred):
    '''Nash-Sutcliffe efficiency, = R^2 pour un predicteur centre.'''
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def stlsq(Theta, y, threshold=0.05, n_iter=15, alpha=1e-8):
    '''Sequential Thresholded Least-Squares (numpy pur, style SINDy).'''
    n_terms = Theta.shape[1]
    XtX = Theta.T @ Theta + alpha * np.eye(n_terms)
    Xty = Theta.T @ y
    coef = np.linalg.solve(XtX, Xty)
    active = np.ones(n_terms, dtype=bool)
    for _ in range(n_iter):
        small = np.abs(coef) < threshold
        if not np.any(small & active):
            break
        active[small] = False
        if not np.any(active):
            break
        Xs = Theta[:, active]
        coef_active = np.linalg.solve(Xs.T @ Xs + alpha * np.eye(Xs.shape[1]), Xs.T @ y)
        coef = np.zeros(n_terms)
        coef[active] = coef_active
    return coef, active


In [ ]:

Theta, names = build_library()
y = flux.ravel()

mu, sigma = Theta.mean(axis=0), Theta.std(axis=0) + 1e-12
Theta_n = (Theta - mu) / sigma
y_n = (y - y.mean()) / (y.std() + 1e-12)

coef_n, active = stlsq(Theta_n, y_n, threshold=0.08)
coef = coef_n * (y.std() / sigma)

print('Termes retenus :')
for name, c, a in zip(names, coef, active):
    if a:
        print(f'  {name:10s} : {c:+.4e}')

r2 = nse(y_n, Theta_n @ coef_n)
print(f'\nNSE (normalisé) = {r2:.3f}')


## 3. Robustesse du split train/test dans le temps

Le split actuel (train/test sur différents pas de temps pour deviner le flux de Reynolds qui dérive dans le temps) est optimiste : un split aléatoire mélange des instants voisins fortement corrélés entre train et test. On compare **walk-forward strict** (blocs temporels ordonnés) vs **split aléatoire**, en numpy pur (`lstsq`), sans nouvelle dépendance. On ajoute une régression **ridge** (forme fermée) comme deuxième technique — dans la continuité du Lasso/OLS déjà recodés à la main.

In [ ]:

def ols_fit_predict(X_train, y_train, X_test):
    coef, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)
    return X_test @ coef


def ridge_fit_predict(X_train, y_train, X_test, lam=1.0):
    n_terms = X_train.shape[1]
    coef = np.linalg.solve(X_train.T @ X_train + lam * np.eye(n_terms), X_train.T @ y_train)
    return X_test @ coef


def walk_forward_splits(n_samples, n_splits=5):
    '''Blocs ordonnés, jamais de fuite du futur vers le passe.'''
    fold = n_samples // (n_splits + 1)
    for i in range(1, n_splits + 1):
        train_idx = np.arange(0, i * fold)
        test_idx  = np.arange(i * fold, min((i + 1) * fold, n_samples))
        if len(test_idx) == 0:
            continue
        yield train_idx, test_idx


def random_splits(n_samples, n_splits=5, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n_samples)
    fold = n_samples // n_splits
    for i in range(n_splits):
        test_idx = idx[i * fold:(i + 1) * fold]
        train_idx = np.setdiff1d(idx, test_idx)
        yield train_idx, test_idx


In [ ]:

# --- Benchmark sur un niveau fixe (z proche du milieu de la couche nuageuse) ---
iz_bench = n_z // 3
X_t = np.stack([ubar[iz_bench], dudz[iz_bench], wbar[iz_bench]], axis=1)  # (n_stat, 3)
y_t = flux[iz_bench]

for name, splitter in [('walk-forward', walk_forward_splits), ('aléatoire', random_splits)]:
    scores_ols, scores_ridge = [], []
    for train_idx, test_idx in splitter(n_stat, n_splits=5):
        pred_ols   = ols_fit_predict(X_t[train_idx], y_t[train_idx], X_t[test_idx])
        pred_ridge = ridge_fit_predict(X_t[train_idx], y_t[train_idx], X_t[test_idx], lam=1.0)
        scores_ols.append(nse(y_t[test_idx], pred_ols))
        scores_ridge.append(nse(y_t[test_idx], pred_ridge))
    print(f'{name:12s} | OLS   NSE moyen = {np.mean(scores_ols):+.3f}'
          f' | Ridge NSE moyen = {np.mean(scores_ridge):+.3f}')

print('\nEcart walk-forward vs aléatoire = biais optimiste du split aléatoire.')


In [ ]:

drift, roll_mean = flux[iz_bench], None
window = max(5, n_stat // 6)
roll_mean = np.convolve(flux[iz_bench], np.ones(window) / window, mode='valid')
drift_rel = (flux[iz_bench, -window:].mean() - flux[iz_bench, :window].mean())
drift_rel /= (np.abs(flux[iz_bench, :window].mean()) + 1e-12)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(flux[iz_bench], alpha=0.4, label='signal brut')
ax.plot(np.arange(window // 2, window // 2 + len(roll_mean)), roll_mean, lw=2,
        label=f'moyenne glissante ({window})')
ax.legend()
ax.set_title(f'Dérive relative début->fin (z={alt[iz_bench]/1000:.1f} km) : {drift_rel:+.1%}')
fig.tight_layout(); plt.show()


## 4. $\psi$ moyennée en $y$ → POD

On calcule $\psi(x,z,t)$ (moyenne-$y$) via le solveur de Poisson périodique-$x$/Dirichlet-$z$ déjà utilisé (notebook lignes de courant), puis une POD par **méthode des snapshots** (`Amat @ Amat.T`, `eigh`) — exactement la méthode déjà validée sur `6_CMT_trois_objectifs`, pas de sklearn.

$$\psi(x,z,t) \approx \bar\psi(x,z) + \sum_n a_n(t)\,\psi_n(x,z), \qquad \tilde u_N=\partial_z\tilde\psi_N/\rho_0,\ \ \tilde w_N=-\partial_x\tilde\psi_N/\rho_0$$

In [ ]:

def psi_poisson_periodic(U2d, W2d, xv, zv, rho0v):
    '''nabla^2 psi = d_x(rho0 w) - d_z(rho0 u), periodique en x, Dirichlet en z.'''
    nz, nx = U2d.shape
    dxl = xv[1] - xv[0]
    rW = rho0v[:, None] * W2d
    rU = rho0v[:, None] * U2d
    dxrW = (np.roll(rW, -1, axis=1) - np.roll(rW, 1, axis=1)) / (2 * dxl)
    dzrU = np.gradient(rU, zv, axis=0)
    omega = dxrW - dzrU

    main_x = -2 * np.ones(nx); off_x = np.ones(nx - 1)
    Lx = diags([main_x, off_x, off_x, [1], [1]], [0, 1, -1, nx - 1, -(nx - 1)]) / dxl ** 2
    dz_mean = np.mean(np.diff(zv))
    main_z = -2 * np.ones(nz); off_z = np.ones(nz - 1)
    Lz = diags([main_z, off_z, off_z], [0, 1, -1]) / dz_mean ** 2
    A = (kron(identity(nz), Lx) + kron(Lz, identity(nx))).tolil()
    b = omega.ravel().copy()

    psi_top = np.cumsum(rho0v * U2d.mean(axis=1)) * dz_mean
    for i in range(nx):
        k0 = i;               A.rows[k0] = [k0]; A.data[k0] = [1.0]; b[k0] = 0.0
        k1 = (nz - 1) * nx + i; A.rows[k1] = [k1]; A.data[k1] = [1.0]; b[k1] = psi_top[-1]
    psi = spsolve(A.tocsr(), b).reshape(nz, nx)
    return psi


In [ ]:

# --- construction des snapshots psi(x,z,t) moyenne-y, sous-echantillonnage temporel raisonnable ---
STRIDE_T = max(1, n_stat // 60)     # ~60 instants au plus pour rester rapide
t_idx_pod = np.arange(0, n_stat, STRIDE_T)

psi_snaps = np.zeros((len(t_idx_pod), n_z, n_x))
ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for i_snap, it in enumerate(t_idx_pod):
    it_global = t_stat + it
    U2d = ds_u['ua'].isel({dim_t: it_global}).mean(dim=dim_y).values   # (nz, nx)
    W2d = ds_w['wa'].isel({dim_t: it_global}).mean(dim=dim_y).values
    psi_snaps[i_snap] = psi_poisson_periodic(U2d, W2d, x, alt, rho0)
ds_u.close(); ds_w.close(); gc.collect()
print('psi_snaps :', psi_snaps.shape)


In [ ]:

# --- POD methode des snapshots (centree !) ---
N = psi_snaps.shape[0]
fmean = psi_snaps.mean(axis=0)
Amat = (psi_snaps - fmean).reshape(N, -1)          # ANOMALIES centrees

Cs = Amat @ Amat.T / N
evals, evecs = np.linalg.eigh(Cs)
order = evals.argsort()[::-1]
evals = np.clip(evals[order], 0, None)
evecs = evecs[:, order]

modes_flat = evecs.T @ Amat                          # (N, nz*nx)
norms = np.linalg.norm(modes_flat, axis=1) + 1e-30
modes_flat = modes_flat / norms[:, None]
modes = modes_flat.reshape(N, n_z, n_x)
a_n = Amat @ modes_flat.T                             # (N_t, N_modes) coefficients temporels

energy = evals / evals.sum()
cum_energy = np.cumsum(energy)
n_modes = int(np.searchsorted(cum_energy, 0.90) + 1)
print(f'{n_modes} modes pour 90% de l energie (sur {len(cum_energy)} modes).')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(cum_energy) + 1), cum_energy, 'o-', ms=3)
ax.axhline(0.90, color='grey', ls='--', lw=0.8)
ax.axvline(n_modes, color='grey', ls='--', lw=0.8)
ax.set_xlabel('nombre de modes'); ax.set_ylabel('énergie cumulée')
ax.set_title('Critère de troncature POD')
fig.tight_layout(); plt.show()


In [ ]:

def reconstruct_uw(n1, n2):
    '''u~_N = d(psi_N)/dz / rho0,  w~_N = -d(psi_N)/dx / rho0, pour les modes n1..n2.'''
    psi_N = fmean[None, :, :] + np.einsum('tn,nzx->tzx', a_n[:, n1:n2 + 1], modes[n1:n2 + 1])
    u_N = np.gradient(psi_N, alt, axis=1) / rho0[None, :, None]
    w_N = -np.gradient(psi_N, x, axis=2) / rho0[None, :, None]
    return psi_N, u_N, w_N


psi_N, u_N, w_N = reconstruct_uw(0, n_modes - 1)

up_N = u_N - u_N.mean(axis=(0, 2), keepdims=True)
wp_N = w_N - w_N.mean(axis=(0, 2), keepdims=True)
flux_N_profile = np.mean(up_N * wp_N, axis=(0, 2)) * rho0

# comparaison au flux complet (3D, meme sous-echantillon temporel)
flux_true_profile = flux[:, t_idx_pod].mean(axis=1)
nse_recon = nse(flux_true_profile, flux_N_profile)
print(f'NSE reconstruction flux (via {n_modes} modes POD) = {nse_recon:.3f}')
print('(rappel : psi est moyenne-y, donc cette reconstruction ne capture que la')
print(' partie organisee T1 — coherent avec le residu T2 deja identifie ailleurs.)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(flux_true_profile, alt / 1000, label='flux complet (3D)')
ax.plot(flux_N_profile, alt / 1000, '--', label=f'{n_modes} modes POD')
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r"$\rho_0\langle u'w'\rangle$"); ax.set_ylabel('z (km)')
ax.legend(); ax.set_title('Reconstruction POD du flux')
fig.tight_layout(); plt.show()


In [ ]:

def mode_energy_content(n1, n2):
    '''psi1^2 = (1/Delta t) sum_t sum_n a_n(t)^2 (modes normes -> ||psi_n||^2=1).'''
    a_block = a_n[:, n1:n2 + 1]
    return np.sum(a_block ** 2) / a_block.shape[0]

for (n1, n2) in [(0, 0), (0, 2), (0, n_modes - 1)]:
    e = mode_energy_content(n1, n2)
    print(f'contenu energetique modes {n1}-{n2} : {e:.4e}')


## 5. Circulation — cellules fermées de $\psi$

Une circulation fermée a un **signe unique** de $\psi'$ le long de son contour. On segmente donc directement par signe constant (`scipy.ndimage.label`, déjà utilisé dans le notebook de détection de tourbillons), puis on calcule $\Gamma = \iint \omega_y\, dx\,dz$ par cellule (Stokes), avec $\omega_y = \partial_z\bar u - \partial_x\bar w$ (vorticité, pas la source du solveur $\psi$).

In [ ]:

def vorticity_uw(U2d, W2d, xv, zv):
    dudz = np.gradient(U2d, zv, axis=0)
    dwdx = np.gradient(W2d, xv, axis=1)
    return dudz - dwdx


def circulation_cells(psi_a, omega, dxl, dzl, min_size=9):
    sign_map = np.where(psi_a >= 0, 1, -1)
    cells = []
    for s in (1, -1):
        mask = sign_map == s
        lbl, n_comp = ndi_label(mask)
        for k in range(1, n_comp + 1):
            comp_mask = lbl == k
            if comp_mask.sum() < min_size:
                continue
            gamma = omega[comp_mask].sum() * dxl * dzl
            cz, cx = np.argwhere(comp_mask).mean(axis=0)
            cells.append(dict(sign=s, area=int(comp_mask.sum()), gamma=gamma, centroid=(cz, cx)))
    cells.sort(key=lambda c: -abs(c['gamma']))
    return cells, sign_map


In [ ]:

# --- application a l'instant central du sous-echantillon POD ---
it_mid = t_idx_pod[len(t_idx_pod) // 2]
psi_mid = psi_snaps[len(t_idx_pod) // 2]
psi_a_mid = psi_mid - psi_mid.mean()

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
U2d_mid = ds_u['ua'].isel({dim_t: t_stat + it_mid}).mean(dim=dim_y).values
W2d_mid = ds_w['wa'].isel({dim_t: t_stat + it_mid}).mean(dim=dim_y).values
ds_u.close(); ds_w.close(); gc.collect()

omega_mid = vorticity_uw(U2d_mid, W2d_mid, x, alt)
dz_mean = np.mean(np.diff(alt))
cells, sign_map = circulation_cells(psi_a_mid, omega_mid, dx, dz_mean, min_size=9)
print(f'{len(cells)} cellules de circulation détectées.')
for c in cells[:8]:
    cz, cx = c['centroid']
    print(f"  signe={c['sign']:+d}  aire={c['area']:5d} pts  "
          f"Gamma={c['gamma']:+.3e}  centre=(z={alt[int(cz)]/1000:.1f}km, x={x[int(cx)]/1000:.0f}km)")


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
vmax = np.nanpercentile(np.abs(psi_a_mid), 98) + 1e-30
im = ax.pcolormesh(x / 1000, alt / 1000, psi_a_mid, cmap='RdBu_r',
                    vmin=-vmax, vmax=vmax, shading='auto')
for c in cells[:10]:
    cz, cx = c['centroid']
    ax.plot(x[int(cx)] / 1000, alt[int(cz)] / 1000, 'ko', ms=4)
    ax.annotate(f"{c['gamma']:.1e}", (x[int(cx)] / 1000, alt[int(cz)] / 1000), fontsize=7)
ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
ax.set_title(f"Cellules de circulation (top {min(10,len(cells))} par |Gamma|)")
fig.colorbar(im, ax=ax, label=r"$\psi'$")
fig.tight_layout(); plt.show()


## Synthèse

- **Obj. 1** : dashboard + Hovmöller prêts.
- **Obj. 2** : STLSQ numpy pur sur bibliothèque grande échelle.
- **Obj. 3** : écart quantifié entre walk-forward et split aléatoire (NSE) — à documenter comme point de vigilance.
- **Obj. 4** : POD par méthode des snapshots (centrée), critère 90% d'énergie, reconstruction du flux validée par NSE — cohérent avec le résidu T2 déjà identifié (ψ moyenne-y ne capture que la partie organisée).
- **Obj. 5** : segmentation par signe constant de ψ, circulation par cellule via Stokes — comparer qualitativement à la détection par extrema+ellipse déjà en place.

Tout le notebook n'utilise que numpy / scipy / xarray / matplotlib — rien de nouveau par rapport à tes notebooks précédents.
